# Preparación del Dataset de Pokemon

Este notebook se encarga de descargar, limpiar y preparar los datos para entrenar una red neuronal que clasifique Pokemon según su tipo principal.

### 1. Instalación de Dependencias
Instalamos la librería `kagglehub` para facilitar la descarga de los datasets desde Kaggle.

In [1]:
!pip install -q kagglehub[pandas-datasets]

### 2. Importación de Librerías
Cargamos las librerías necesarias para el procesamiento de datos, manejo de imágenes y entrenamiento con PyTorch.

In [2]:
import os
import json
import random
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import kagglehub
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

### 3. Configuración de Credenciales de Kaggle
Configuramos las credenciales necesarias para acceder a la API de Kaggle.

In [3]:
KAGGLE_USERNAME = "tomsguiazu"
KAGGLE_KEY = "KGAT_bf485502d284e7837b119eb874b76139"

os.makedirs("/root/.kaggle", exist_ok=True)

kaggle_credentials = {
    "username": KAGGLE_USERNAME,
    "key": KAGGLE_KEY
}

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_credentials, f)

!chmod 600 /root/.kaggle/kaggle.json

print("Kaggle configurado correctamente")

Kaggle configurado correctamente


### 4. Descarga y Extracción de Datasets
Descargamos los datasets de información tabular (CSV) y de imágenes de Pokemon.

In [4]:
# Descargar dataset tabular
!curl -L -o pokemon.zip https://www.kaggle.com/api/v1/datasets/download/mlomuscio/pokemon
!unzip -q pokemon.zip -d pokemon_dataset

# Descargar dataset de imágenes
!curl -L -o pokemon_images.zip https://www.kaggle.com/api/v1/datasets/download/lantian773030/pokemonclassification
!unzip -q pokemon_images.zip -d pokemon_images_dataset

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 14361  100 14361    0     0  32907      0 --:--:-- --:--:-- --:--:-- 32907
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  417M  100  417M    0     0  96.0M      0  0:00:04  0:00:04 --:--:-- 95.3M


### 5. Carga y Limpieza de Datos Tabulares
Cargamos el CSV, filtramos por la Generación 1 y normalizamos los nombres para que coincidan con las carpetas de imágenes.

In [5]:
CSV_PATH = "pokemon_dataset/PokemonData.csv"
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.lower()

# Filtrar solo Generación 1
df_gen1 = df[df["generation"] == 1][["name", "type1"]]

def normalize_name(name):
    return str(name).lower().replace(".", "").replace("'", "").replace("♀", "").replace("♂", "").replace(" ", "").replace("-", "")

df_gen1["normalized_name"] = df_gen1["name"].apply(normalize_name)

# Corregir nombres específicos y eliminar duplicados o errores
df_gen1["name"] = df_gen1["name"].replace({"Mr. Mime": "MrMime", "Farfetch'd": "Farfetchd"})
df_gen1 = df_gen1[~df_gen1["name"].isin(["Nidoranâ™€", "Nidoranâ™‚"])]
df_gen1 = df_gen1[~df_gen1["name"].str.contains("Mega", na=False)]

print(f"Cantidad de Pokemon luego de limpieza: {len(df_gen1)}")

Cantidad de Pokemon luego de limpieza: 149


### 6. Mapeo de Imágenes con Tipos
Recorremos las carpetas de imágenes y las asociamos con su tipo principal definido en el CSV.

In [6]:
IMAGES_PATH = "pokemon_images_dataset/PokemonData"
image_data = []

for folder_name in os.listdir(IMAGES_PATH):
    folder_path = os.path.join(IMAGES_PATH, folder_name)
    if not os.path.isdir(folder_path): continue

    normalized_folder = normalize_name(folder_name)
    pokemon_match = df_gen1[df_gen1["normalized_name"] == normalized_folder]

    if pokemon_match.empty: continue

    pokemon_name = pokemon_match.iloc[0]["name"]
    pokemon_type = pokemon_match.iloc[0]["type1"]

    VALID_EXTENSIONS = (".jpg", ".jpeg", ".png")
    for image_name in os.listdir(folder_path):
        if not image_name.lower().endswith(VALID_EXTENSIONS): continue
        image_path = os.path.join(folder_path, image_name)
        image_data.append({"pokemon": pokemon_name, "type1": pokemon_type, "image_path": image_path})

final_df = pd.DataFrame(image_data)
print(f"Total de imágenes encontradas: {len(final_df)}")

Total de imágenes encontradas: 6779


### 7. Definición de Transformaciones e Índices de Clase
Preparamos las transformaciones (Data Augmentation para entrenamiento) y creamos el mapeo de tipos a índices numéricos.

In [7]:
# Transformaciones
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Mapeo de clases
classes = sorted(final_df["type1"].unique())
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
idx_to_class = {v: k for k, v in class_to_idx.items()}

print("Mapeo de clases:", class_to_idx)

Mapeo de clases: {'Bug': 0, 'Dragon': 1, 'Electric': 2, 'Fairy': 3, 'Fighting': 4, 'Fire': 5, 'Ghost': 6, 'Grass': 7, 'Ground': 8, 'Ice': 9, 'Normal': 10, 'Poison': 11, 'Psychic': 12, 'Rock': 13, 'Water': 14}


### 8. Clase Custom Dataset de PyTorch
Definimos una clase para cargar las imágenes y sus etiquetas de forma eficiente.

In [8]:
class PokemonTypeDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = class_to_idx[row["type1"]]
        return image, label

### 9. División del Dataset
Dividimos los datos en conjuntos de Entrenamiento (70%), Validación (15%) y Prueba (15%) de forma estratificada por tipo.

In [9]:
SEED = 43
train_df, temp_df = train_test_split(final_df, test_size=0.30, stratify=final_df["type1"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["type1"], random_state=SEED)

train_dataset = PokemonTypeDataset(train_df, transform=train_transform)
val_dataset = PokemonTypeDataset(val_df, transform=eval_transform)
test_dataset = PokemonTypeDataset(test_df, transform=eval_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

print("TRAIN")
print(train_df["type1"].value_counts())

print("\nVALIDATION")
print(val_df["type1"].value_counts())

print("\nTEST")
print(test_df["type1"].value_counts())

images, labels = next(
    iter(train_loader)
)

Train: 4745, Val: 1017, Test: 1017
TRAIN
type1
Water       918
Normal      688
Grass       409
Poison      381
Fire        375
Bug         374
Electric    314
Rock        263
Psychic     243
Ground      239
Fighting    236
Ghost        99
Dragon       81
Ice          66
Fairy        59
Name: count, dtype: int64

VALIDATION
type1
Water       197
Normal      147
Grass        87
Poison       82
Bug          80
Fire         80
Electric     67
Rock         57
Psychic      52
Ground       52
Fighting     51
Ghost        21
Dragon       17
Ice          14
Fairy        13
Name: count, dtype: int64

TEST
type1
Water       197
Normal      148
Grass        88
Poison       81
Fire         81
Bug          80
Electric     68
Rock         56
Psychic      52
Ground       51
Fighting     50
Ghost        21
Dragon       18
Ice          14
Fairy        12
Name: count, dtype: int64


## 10. Modelo preentrenado y estrategia de fine-tuning

Para el entrenamiento se utilizó una red ResNet18 preentrenada sobre ImageNet. Esta arquitectura fue elegida porque permite aprovechar filtros ya aprendidos en un dataset grande, como bordes, texturas, colores y formas generales.

Como nuestro dataset de Pokémon no es muy grande, no se entrenó toda la red desde cero. En su lugar, se aplicó fine-tuning parcial: se congelaron las capas iniciales de ResNet18 y se reemplazó la capa final para adaptarla a la cantidad de clases del problema.

La capa final original de ResNet18 clasifica 1000 clases de ImageNet, por lo tanto fue reemplazada por una nueva capa lineal cuya salida coincide con la cantidad de tipos principales de Pokémon.


In [75]:
from torchvision import models
import torch.nn as nn
import torch.optim as optim

# Entrenamos con GPU o CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo utilizado:", device)

# Cantidad de clases del problema
num_classes = len(classes)

print("Clases:", classes)
print("Cantidad de clases:", num_classes)

##Crear Modelos
def crear_modelo_resnet18():
  # Cargar ResNet18 preentrenada en ImageNet
  model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

  # Congelar todas las capas preentrenadas
  for param in model.parameters():
      param.requires_grad = False

  # Reemplazar la capa final
  num_features = model.fc.in_features ##cantidad de entradas de la capa final original
  model.fc = nn.Linear(num_features, num_classes)

  # Enviar modelo al dispositivo
  model = model.to(device)
  return model

model = crear_modelo_resnet18()

print(model)


Dispositivo utilizado: cuda
Clases: ['Bug', 'Dragon', 'Electric', 'Fairy', 'Fighting', 'Fire', 'Ghost', 'Grass', 'Ground', 'Ice', 'Normal', 'Poison', 'Psychic', 'Rock', 'Water']
Cantidad de clases: 15
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2

La estrategia aplicada fue **fine-tuning** sobre una **ResNet18 preentrenada en ImageNet**.

En una primera etapa se congelaron las capas preentrenadas de ResNet18, principalmente las capas convolucionales iniciales e intermedias. Estas capas ya habían aprendido características generales en ImageNet, como bordes, colores, texturas y formas simples, que también son útiles para clasificar imágenes de Pokémon.

La capa que sí se entrenó desde el comienzo fue la **capa final** (`fc`), ya que fue reemplazada para adaptarse a la cantidad de clases de nuestro problema. La ResNet18 original clasifica 1000 clases de ImageNet, por lo tanto fue necesario cambiar esa salida por una nueva capa lineal con la cantidad de tipos principales de Pokémon del dataset.

No se entrenó todo el modelo **end-to-end** desde el inicio. Esto se decidió porque el dataset no es lo suficientemente grande como para ajustar todos los parámetros de una red profunda sin riesgo de **overfitting**.


##Calculo del valor del loss y accuracy

In [76]:
def evaluate_model(model, val_loader, loss, device):
    model.eval()
    L = 0.0
    N = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)

            y_hat = model(X)
            l = loss(y_hat, y)

            L += l.sum().item()
            N += l.numel()

            preds = y_hat.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.numel()

    val_loss = L / N
    val_acc = correct / total

    return val_loss, val_acc

##Entrenamiento

###Por las dudas

In [43]:
## Este modelo seria sin la tabla comparativa de abajo y sin la linea de arriba
##def train_fine_tuning(model, learning_rate, batch_size=BATCH_SIZE, num_epochs=5,
                      param_group=True, optimizer_name="sgd", scheduler_name=None):

    train_iter = train_loader

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loss = nn.CrossEntropyLoss(reduction="none")

    if optimizer_name == "sgd":
        if param_group:
            params_1x = [
                param for name, param in model.named_parameters()
                if name not in ["fc.weight", "fc.bias"]
            ]

            trainer = torch.optim.SGD(
                [
                    {'params': params_1x},
                    {'params': model.fc.parameters(), 'lr': learning_rate * 10}
                ],
                lr=learning_rate,
                weight_decay=0.001
            )
        else:
            trainer = torch.optim.SGD(
                model.parameters(),
                lr=learning_rate,
                weight_decay=0.001
            )

    elif optimizer_name == "adam":
        trainer = torch.optim.Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=0.001
        )

    elif optimizer_name == "adamw":
        trainer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=0.01
        )

    else:
        raise ValueError("Optimizador no reconocido")

    if scheduler_name == "plateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            trainer,
            mode="min",
            factor=0.1,
            patience=2
        )
    else:
        scheduler = None

    model.to(device)

    historial = {
        "loss": []
    }

    for epoch in range(num_epochs):
        model.train()
        L = 0.0
        N = 0

        for X, y in train_iter:
            X, y = X.to(device), y.to(device)

            l = loss(model(X), y)

            trainer.zero_grad()
            l.sum().backward()
            trainer.step()

            L += l.sum().item()
            N += l.numel()

        epoch_loss = L / N
        historial["loss"].append(epoch_loss)

        if scheduler is not None:
            scheduler.step(epoch_loss)

        print(f'epoch {epoch + 1}, loss {epoch_loss:f}')

    return historial

###Correcto

In [77]:
def train_fine_tuning(model, learning_rate, batch_size=BATCH_SIZE, num_epochs=5,
                      param_group=True, optimizer_name="sgd", scheduler_name=None):

    train_iter = train_loader

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loss = nn.CrossEntropyLoss(reduction="none")

    if optimizer_name == "sgd":
        if param_group:
            params_1x = [
                param for name, param in model.named_parameters()
                if name not in ["fc.weight", "fc.bias"]
            ]

            trainer = torch.optim.SGD(
                [
                    {'params': params_1x},
                    {'params': model.fc.parameters(), 'lr': learning_rate * 10}
                ],
                lr=learning_rate,
                weight_decay=0.001
            )
        else:
            trainer = torch.optim.SGD(
                model.parameters(),
                lr=learning_rate,
                weight_decay=0.001
            )

    elif optimizer_name == "adam":
        trainer = torch.optim.Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=0.001
        )

    elif optimizer_name == "adamw":
        trainer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=0.01
        )

    elif optimizer_name == "sgd_momentum":
        trainer = torch.optim.SGD(
            model.parameters(),
            lr=learning_rate,
            momentum=0.9,
            weight_decay=0.001
        )

    else:
        raise ValueError("Optimizador no reconocido")

    if scheduler_name == "plateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            trainer,
            mode="min",
            factor=0.1,
            patience=2
        )
    else:
        scheduler = None

    model.to(device)

    historial = {
        "train_loss": [],
        "val_loss": [],
        "val_acc": []
    }

    for epoch in range(num_epochs):
        model.train()
        L = 0.0
        N = 0

        for X, y in train_iter:
            X, y = X.to(device), y.to(device)

            l = loss(model(X), y)

            trainer.zero_grad()
            l.sum().backward()
            trainer.step()

            L += l.sum().item()
            N += l.numel()

        train_loss = L / N
        val_loss, val_acc = evaluate_model(model, val_loader, loss, device)

        if scheduler is not None:
            scheduler.step(val_loss)

        historial["train_loss"].append(train_loss)
        historial["val_loss"].append(val_loss)
        historial["val_acc"].append(val_acc)

        print(f'epoch {epoch + 1}, train loss {train_loss:.4f}, val loss {val_loss:.4f}, val acc {val_acc:.4f}')

    return historial

##Configuración 1 — SGD


In [79]:

model1 = crear_modelo_resnet18()
config_1 = train_fine_tuning(
    model1,
    learning_rate=5e-5,
    batch_size=BATCH_SIZE,
    num_epochs=10,
    param_group=True,
    optimizer_name="sgd",
    scheduler_name=None
)

epoch 1, train loss 2.1185, val loss 1.8360, val acc 0.4326
epoch 2, train loss 1.6162, val loss 1.4731, val acc 0.5202
epoch 3, train loss 1.3832, val loss 1.3517, val acc 0.5575
epoch 4, train loss 1.2541, val loss 1.2614, val acc 0.5870


##Configuración 2 — SGD sin grupos de parametros


In [80]:
model2 = crear_modelo_resnet18()

config_2 = train_fine_tuning(
    model2,
    learning_rate=5e-5,
    batch_size=BATCH_SIZE,
    num_epochs=10,
    param_group=False,
    optimizer_name="sgd",
    scheduler_name=None
)

epoch 1, train loss 2.4696, val loss 2.3866, val acc 0.2262
epoch 2, train loss 2.3022, val loss 2.2638, val acc 0.2734
epoch 3, train loss 2.1911, val loss 2.1522, val acc 0.3156
epoch 4, train loss 2.0842, val loss 2.0548, val acc 0.3559


##Configuración 3: AdamW con scheduler

In [81]:
model3 = crear_modelo_resnet18()

config_3 = train_fine_tuning(
    model3,
    learning_rate=1e-4,
    batch_size=BATCH_SIZE,
    num_epochs=8,
    param_group=False,
    optimizer_name="adamw",
    scheduler_name="plateau"
)

epoch 1, train loss 2.4915, val loss 2.3652, val acc 0.2321
epoch 2, train loss 2.3135, val loss 2.2262, val acc 0.2763
epoch 3, train loss 2.1769, val loss 2.0957, val acc 0.3235
epoch 4, train loss 2.0535, val loss 2.0044, val acc 0.3589


##Tabla comparativa de experimentos

In [82]:
import pandas as pd

tabla_experimentos = pd.DataFrame({
    "Configuración": [
        "Config. 1 - SGD con LR diferenciado",
        "Config. 2 - SGD sin LR diferenciado",
        "Config. 3 - AdamW + scheduler"
    ],
    "Optimizador": [
        "SGD",
        "SGD",
        "AdamW"
    ],
    "Learning rate": [
        "5e-5 / fc: 5e-4",
        "5e-5",
        "1e-4"
    ],
    "Scheduler": [
        "No",
        "No",
        "ReduceLROnPlateau"
    ],
    "Épocas": [
        10,
        10,
        8
    ],
    "Mejor val accuracy": [
        max(config_1["val_acc"]),
        max(config_2["val_acc"]),
        max(config_3["val_acc"])
    ],
    "Menor val loss": [
        min(config_1["val_loss"]),
        min(config_2["val_loss"]),
        min(config_3["val_loss"])
    ]
})

tabla_experimentos

,Configuración,Optimizador,Learning rate,Scheduler,Épocas,Mejor val accuracy,Menor val loss
0,Config. 1 - SGD con LR diferenciado,SGD,5e-5 / fc: 5e-4,No,10,0.587021,1.261403
1,Config. 2 - SGD sin LR diferenciado,SGD,5e-5,No,10,0.355949,2.054758
2,Config. 3 - AdamW + scheduler,AdamW,1e-4,ReduceLROnPlateau,8,0.358899,2.004363


##Configuración 4 — SGD con momentum y learning rate bajo

In [74]:
model4 = crear_modelo_resnet18()

config_4 = train_fine_tuning(
    model4,
    learning_rate=1e-4,
    batch_size=BATCH_SIZE,
    num_epochs=10,
    param_group=False,
    optimizer_name="sgd_momentum",
    scheduler_name=None
)

epoch 1, train loss 0.9289, val loss 1.1147, val acc 0.6401
epoch 2, train loss 0.8972, val loss 1.0087, val acc 0.6667
epoch 3, train loss 0.8867, val loss 1.0127, val acc 0.6578
epoch 4, train loss 0.8437, val loss 1.0335, val acc 0.6500
epoch 5, train loss 0.8533, val loss 1.0124, val acc 0.6647
epoch 6, train loss 0.8325, val loss 0.9851, val acc 0.6755
epoch 7, train loss 0.7939, val loss 1.0104, val acc 0.6657
epoch 8, train loss 0.8161, val loss 1.0001, val acc 0.6765


KeyboardInterrupt: 